# 08. Cross-venue information: does the futures market add early-warning information?

Changing the alert schedule did not increase hits before the 1% onset (notebook 05), and the gap between the hindsight oracle of notebook 04 (68 events caught, 39 of them before the onset) and the calendar policy (39 caught, 22 before the onset) is large. If the remaining bottleneck is information, same-minute futures-market information might add something to the spot V+F model. I test that here on Binance spot BTCUSDT, with Binance futures as the second venue.

- New features, all causal (data up to minute t only) and fixed in advance:
  - `z_basis`: the 15-minute mean of the futures/spot price gap (log, bps), z-scored against the previous 30 days.
  - `d_flow`: the 15-minute taker sell share on futures minus the same share on spot.
  - `z_vr`: the log futures/spot 15-minute volume ratio, z-scored against the previous 30 days.
- I compare five models (M0 = the existing V+F, M1–M3 = V+F plus one feature, M4 = V+F plus all three) on the same minute set. I train them on 2021, calibrate them on 2022 (calendar policy with 20 alerts per month and 240 per year, the same rule as notebook 05) and evaluate them on 2023 to 2026-04. The primary metric is the same as in notebooks 05–07: a hit 30–240 minutes before the breach and before the 1% onset.
- I read the spot cache (`btc_lead_cost_results_77months_spot/cache`) and the futures cache (`btc_lead_cost_results_77months/cache`) and write the results to `btc_lead_cost_results_77months_spot/crossvenue_information_v1/`.
- This analysis is exploratory, since I have already inspected 2023–2026 several times. The pre-set rule selects no candidate for notebook 11 (the unused 2018–2020 period; cross-venue features from 2019-11). I write the candidate-selection rule into the specification `crossvenue_spec.json` before I see the results.

In [1]:
from pathlib import Path
from collections import deque, defaultdict
import io, os, sys, json, math, time, hashlib, zipfile, warnings, urllib.request, urllib.error
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import binomtest
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
try:
    import pyarrow  # noqa: F401
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'pyarrow'])
try:
    from numba import njit
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'numba'])
    from numba import njit
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab: paths below must point to a folder with the same structure.')

DRIVE = Path('/content/drive/MyDrive')
SPOT_BASE = DRIVE / 'btc_lead_cost_results_77months_spot'      # spot results (notebooks 01–07)
FUT_BASE = DRIVE / 'btc_lead_cost_results_77months'            # futures results, read only
BINANCE = 'https://data.binance.vision/data/'
NS = 60 * 10**9
LEAD_MIN = 30; LEAD_MAX = 240; COOLDOWN = 240; COOLDOWN_MIN = 240
B = 240; MONTH_CAP = 20
EVAL_START = '2023-01-01'; EVAL_END = '2026-05-01'

_src = SPOT_BASE / 'cache' / 'DATA_SOURCE.json'
if not _src.is_file() or json.loads(_src.read_text()).get('venue') != 'spot':
    raise RuntimeError(f'Spot cache not found: {_src}. Run notebook 01 first.')
REF_PATH = SPOT_BASE / 'causal_alert_selection_v1' / 'spot_reference.json'
THR_PATH = SPOT_BASE / 'FINAL_risk_adaptive_alert_capacity_v1' / 'spot_frozen_thresholds.json'
for _p in (REF_PATH, THR_PATH):
    if not _p.is_file():
        raise FileNotFoundError(f'Results of notebooks 05 and 06 are required: {_p}')
REF = json.loads(REF_PATH.read_text())
FROZEN = json.loads(THR_PATH.read_text())
assert REF.get('venue') == 'spot' and FROZEN.get('venue') == 'spot'
TH_CAL = float(REF['TH_CAL']); TH_HIGH = float(REF['TH_HIGH']); TH_BUCKET = float(FROZEN['TH_BUCKET'])
print('SPOT cache verified. Frozen spot thresholds:', dict(TH_CAL=TH_CAL, TH_HIGH=TH_HIGH, TH_BUCKET=TH_BUCKET))


def sha256_file(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def load_btc_spot():
    px = pd.read_parquet(SPOT_BASE / 'cache' / 'BTCUSDT_1m_regularized.parquet').sort_index()
    sg = pd.read_parquet(SPOT_BASE / 'cache' / 'btc_signals_1m.parquet', columns=['z_vol', 'tsr', 'z_hs']).sort_index()
    for fr in (px, sg):
        assert isinstance(fr.index, pd.DatetimeIndex) and fr.index.tz is not None
        fr.index = fr.index.tz_convert('UTC')
    assert px.index.equals(sg.index), 'price/signal caches must share one 1-minute index'
    assert px.index.is_unique and (np.diff(px.index.asi8) == NS).all()
    return px, sg


def load_catalog(path):
    E = pd.read_csv(path)
    for c in ['peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']:
        E[c] = pd.to_datetime(E[c], utc=True)
    E = E.sort_values('breach_utc').reset_index(drop=True)
    assert E.event_id.is_unique
    return E


def label_from_catalog(T, E):
    """1 if a 4% breach occurs 30-240 minutes after minute t (same rule as notebooks 04–07)."""
    b = E.breach_utc.astype('int64').to_numpy()
    pos = np.searchsorted(b, T + LEAD_MIN * NS, 'left')
    y = np.zeros(len(T), dtype=np.int8)
    ok = pos < len(b)
    y[ok] = (b[pos[ok]] <= T[ok] + LEAD_MAX * NS).astype(np.int8)
    return y


def ts(s):
    return pd.Timestamp(s, tz='UTC')


def in_period(T, start, end):
    return (T >= ts(start).value) & (T < ts(end).value)


def train_mask(T, valid, year=2021):
    return valid & (T >= ts(f'{year}-01-01').value) & (T < ts(f'{year + 1}-01-01').value - (LEAD_MAX + 5) * NS)


def new_logit():
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.2, class_weight='balanced', max_iter=400,
                                                              solver='lbfgs', random_state=17))


def events_between(E, start, end):
    """Evaluation events: breach in [start+4h, end) so every warning window lies inside the period."""
    return E[(E.breach_utc >= ts(start) + pd.Timedelta(minutes=LEAD_MAX)) & (E.breach_utc < ts(end))].reset_index(drop=True)


# --- emit_calendar, as in notebook 05 ---
def emit_calendar(t,eligible,B=240,cap=20,cooldown_min=240):
    t=np.asarray(t,dtype=np.int64)
    eligible=np.asarray(eligible,dtype=bool)
    assert len(t)==len(eligible) and np.all(np.diff(t)>0)
    ids=np.flatnonzero(eligible)
    z=[]; mcount=0; ycount=0; pm=None; py=None; last=-(10**20)
    for k in ids:
        ti=int(t[k]); dt=pd.Timestamp(ti,tz='UTC')
        year=int(dt.year); month=year*12+int(dt.month)
        if year!=py: ycount=0; py=year
        if month!=pm: mcount=0; pm=month
        if ycount>=B or mcount>=cap or ti-last<cooldown_min*NS:continue
        z.append(ti); last=ti; ycount+=1; mcount+=1
    return np.asarray(z,dtype=np.int64)

# --- match_events, as in notebook 05 ---
def match_events(events,alerts):
    alerts=np.sort(np.asarray(alerts,dtype=np.int64))
    out=[];used=set()
    for ev in events.itertuples(index=False):
        breach=ev.breach_utc.value; onset=ev.onset_1pct_utc.value
        left=int(np.searchsorted(alerts,breach-LEAD_MAX*NS,'left'))
        right=int(np.searchsorted(alerts,breach-LEAD_MIN*NS,'right'))
        for j in range(left,right):
            if j not in used:
                used.add(j)
                out.append(dict(event_id=int(ev.event_id),breach_utc=ev.breach_utc,
                                alarm_utc=pd.Timestamp(int(alerts[j]),tz='UTC'),
                                lead_min=(breach-int(alerts[j]))/NS,
                                pre1pct=bool(alerts[j]<onset)))
                break
    return pd.DataFrame(out,columns=['event_id','breach_utc','alarm_utc','lead_min','pre1pct'])

# --- statistics, as in notebook 05 ---
def statistics(events,alerts):
    hits=match_events(events,alerts)
    n=len(alerts);N=len(events)
    return dict(alarms=n,caught=len(hits),before_1pct=int(hits.pre1pct.sum()),
                false_alarms=n-len(hits),recall=len(hits)/N,
                pre_onset_recall=int(hits.pre1pct.sum())/N,
                alert_precision=(len(hits)/n if n else 0))



def cal_fields(t):
    idx = pd.to_datetime(np.asarray(t, dtype='int64'), utc=True)
    years = idx.year.to_numpy().astype(np.int64)
    months = (idx.year.to_numpy() * 12 + idx.month.to_numpy()).astype(np.int64)
    return years, months


@njit(cache=False)
def _calendar_core(t, elig, years, months, B, caps, cooldown_ns):
    # Line-by-line port of emit_calendar with a per-minute monthly cap (caps) so a surge policy can reuse it.
    out = np.empty(len(t), np.int64); k = 0
    mcount = 0; ycount = 0; pm = -1; py = -1; last = -(2**62)
    for i in range(len(t)):
        if not elig[i]:
            continue
        ti = t[i]; yr = years[i]; mo = months[i]
        if yr != py:
            ycount = 0; py = yr
        if mo != pm:
            mcount = 0; pm = mo
        if ycount >= B or mcount >= caps[i] or ti - last < cooldown_ns:
            continue
        out[k] = ti; k += 1; last = ti; ycount += 1; mcount += 1
    return out[:k].copy()


_CAL_CACHE = [None, None]


def emit_calendar_fast(t, eligible, B=240, cap=20, cooldown_min=240, caps=None):
    t = np.asarray(t, dtype=np.int64); eligible = np.asarray(eligible, dtype=bool)
    assert len(t) == len(eligible) and (len(t) < 2 or np.all(np.diff(t) > 0))
    if _CAL_CACHE[0] is t:
        years, months = _CAL_CACHE[1]
    else:
        years, months = cal_fields(t); _CAL_CACHE[0] = t; _CAL_CACHE[1] = (years, months)
    if caps is None:
        caps = np.full(len(t), int(cap), np.int64)
    return _calendar_core(t, eligible, years, months, int(B), np.asarray(caps, np.int64), np.int64(cooldown_min * NS))


def check_fast_calendar(t, s):
    """Stop if the compiled calendar ever differs from the notebook 05 reference implementation."""
    qs = np.unique(np.quantile(s[np.isfinite(s)], [0., .5, .8, .9, .95, .99, .999]))
    for th in qs:
        e = np.isfinite(s) & (s >= th)
        for B_, cap_ in [(240, 20), (60, 5)]:
            if not np.array_equal(emit_calendar(t, e, B=B_, cap=cap_), emit_calendar_fast(t, e, B=B_, cap=cap_)):
                raise AssertionError(f'fast calendar differs from the reference at th={th}, B={B_}')
    return len(qs) * 2


def score_grid(sc):
    Q = np.unique(np.r_[0., np.linspace(.10, .995, 33), .997, .999, .9999, 1.])
    return np.unique(np.r_[np.nextafter(sc.min(), -np.inf), np.quantile(sc, Q), np.nextafter(sc.max(), np.inf)])


def calibrate_calendar(t_cal, s_cal, grid=None, B_=240, cap_=20):
    """Notebook 05 rule: the most 2022 alerts reachable (cap = B), then the fewest raw candidates (strictest)."""
    grid = score_grid(s_cal) if grid is None else grid
    rows = []
    for th in grid:
        e = s_cal >= th
        rows.append((float(th), len(emit_calendar_fast(t_cal, e, B=B_, cap=cap_)), int(e.sum())))
    df = pd.DataFrame(rows, columns=['threshold', 'cal_alerts', 'raw_candidates'])
    best = df[df.cal_alerts == df.cal_alerts.max()].sort_values('raw_candidates').iloc[0]
    return float(best.threshold), int(best.cal_alerts), df


def mcnemar_one_sided(a_only, b_only):
    """P(at least b_only successes | n discordant, p=.5): evidence that B beats A."""
    n = a_only + b_only
    return float(binomtest(b_only, n, .5, alternative='greater').pvalue) if n else 1.0


def mcnemar_two_sided(a_only, b_only):
    n = a_only + b_only
    return float(binomtest(min(a_only, b_only), n, .5).pvalue) if n else 1.0


def paired(events, base_alerts, new_alerts):
    mb = match_events(events, base_alerts); mn = match_events(events, new_alerts)
    hb, hn = set(mb.event_id), set(mn.event_id)
    pb, pn = set(mb.loc[mb.pre1pct, 'event_id']), set(mn.loc[mn.pre1pct, 'event_id'])
    return dict(base_caught=len(hb), new_caught=len(hn), base_only_caught=len(hb - hn), new_only_caught=len(hn - hb),
                p_caught_two_sided=mcnemar_two_sided(len(hb - hn), len(hn - hb)),
                base_pre1pct=len(pb), new_pre1pct=len(pn), base_only_pre1pct=len(pb - pn), new_only_pre1pct=len(pn - pb),
                p_pre1pct_two_sided=mcnemar_two_sided(len(pb - pn), len(pn - pb)),
                p_pre1pct_one_sided_new_better=mcnemar_one_sided(len(pb - pn), len(pn - pb)))
XV_WIN = 15                      # minutes, pre-declared
ZWIN = 30 * 1440; ZMIN = 15 * 1440  # same trailing 30-day z-score window as z_vol / z_hs
XV_FEATURES = ['z_basis', 'd_flow', 'z_vr']


def zscore_trailing(s):
    r = s.rolling(ZWIN, min_periods=ZMIN)
    return (s - r.mean().shift(1)) / r.std().shift(1).replace(0, np.nan)


def cross_venue_features(spot_px, fut_px):
    """Causal cross-venue features at minute t (both frames: regular UTC 1m grid, columns close/volume/taker_buy_vol).
    z_basis: 15-min mean of log(futures/spot) in bps, z-scored against the previous 30 days.
    d_flow : 15-min taker-sell share on futures minus the same share on spot.
    z_vr   : log(futures/spot 15-min volume), z-scored against the previous 30 days."""
    assert spot_px.index.equals(fut_px.index)
    sc = spot_px.close.where(spot_px.volume > 0); fc = fut_px.close.where(fut_px.volume > 0)
    basis = np.log(fc / sc) * 1e4
    z_basis = zscore_trailing(basis.rolling(XV_WIN, min_periods=XV_WIN).mean())
    sv = spot_px.volume.rolling(XV_WIN, min_periods=XV_WIN).sum()
    fv = fut_px.volume.rolling(XV_WIN, min_periods=XV_WIN).sum()
    ss = (spot_px.volume - spot_px.taker_buy_vol).rolling(XV_WIN, min_periods=XV_WIN).sum()
    fs = (fut_px.volume - fut_px.taker_buy_vol).rolling(XV_WIN, min_periods=XV_WIN).sum()
    d_flow = fs / fv.replace(0, np.nan) - ss / sv.replace(0, np.nan)
    z_vr = zscore_trailing(np.log(fv.replace(0, np.nan) / sv.replace(0, np.nan)))
    return pd.DataFrame({'z_basis': z_basis.astype('float32'), 'd_flow': d_flow.astype('float32'),
                         'z_vr': z_vr.astype('float32')}, index=spot_px.index)


FEATURE_SETS = {
    'M0_VF': ['z_vol', 'tsr'],
    'M1_VF_basis': ['z_vol', 'tsr', 'z_basis'],
    'M2_VF_flowgap': ['z_vol', 'tsr', 'd_flow'],
    'M3_VF_volratio': ['z_vol', 'tsr', 'z_vr'],
    'M4_VF_all3': ['z_vol', 'tsr', 'z_basis', 'd_flow', 'z_vr'],
}

OUT = SPOT_BASE / 'crossvenue_information_v1'
OUT.mkdir(parents=True, exist_ok=True)
SPEC = {
    'study': 'cross-venue information, exploratory', 'train': '2021', 'calibration': '2022 (calendar, 240/yr, 20/month; notebook-5 rule)',
    'evaluation': f'{EVAL_START} to {EVAL_END} exclusive', 'features': XV_FEATURES, 'window_minutes': XV_WIN,
    'models': FEATURE_SETS, 'comparison_mask': 'minutes where spot V+F inputs AND all cross-venue features are valid',
    'primary_endpoint': 'alert 30-240 min before 4% breach and before 1% onset',
    'candidate_rule': 'challenger (M1-M4) with the most pre-onset hits; ties -> fewer features; passed to notebook 11 only if '
                      'strictly more pre-onset hits than M0_VF on the same comparison mask, otherwise no candidate',
    'status': 'exploratory; 2023-2026 previously inspected'}
_raw = (json.dumps(SPEC, indent=2, sort_keys=True) + '\n').encode()
(OUT / 'crossvenue_spec.json').write_bytes(_raw)
(OUT / 'crossvenue_spec.sha256').write_text(hashlib.sha256(_raw).hexdigest() + '  crossvenue_spec.json\n')
print('Output:', OUT)

Mounted at /content/drive
SPOT cache verified. Frozen spot thresholds: {'TH_CAL': 0.6698990591069133, 'TH_HIGH': 0.6864782740093911, 'TH_BUCKET': 0.7166067355142407}
Output: /content/drive/MyDrive/btc_lead_cost_results_77months_spot/crossvenue_information_v1


## 1. Load the spot and futures caches and compute the cross-venue features

In [2]:
px, sg = load_btc_spot()
fut = pd.read_parquet(FUT_BASE / 'cache' / 'BTCUSDT_1m_regularized.parquet', columns=['close', 'volume', 'taker_buy_vol']).sort_index()
fut.index = fut.index.tz_convert('UTC')
assert fut.index.is_unique and (np.diff(fut.index.asi8) == NS).all(), 'futures cache must be a regular 1-minute grid'
print('spot  :', px.index[0], '->', px.index[-1], len(px))
print('futures:', fut.index[0], '->', fut.index[-1], len(fut))
fut = fut.reindex(px.index)
print('futures minutes available on the spot grid:', f'{np.isfinite(fut.close.to_numpy()).mean():.4%}')
xv = cross_venue_features(px[['close', 'volume', 'taker_buy_vol']], fut)

E = load_catalog(SPOT_BASE / 'event_definition_eligibility_audit' / 'price_only_event_catalog.csv')
T = px.index.asi8
close = px.close.to_numpy(float); volume = px.volume.to_numpy(float)
v = sg.z_vol.to_numpy(float); f = sg.tsr.to_numpy(float); h = sg.z_hs.to_numpy(float)
base_valid = (np.isfinite(close) & (close > 0) & np.isfinite(volume) & (volume > 0) &
              np.isfinite(v) & np.isfinite(f) & np.isfinite(h) & (f >= 0) & (f <= 1))
xv_ok = np.isfinite(xv[XV_FEATURES].to_numpy(float)).all(axis=1)
common = base_valid & xv_ok
y = label_from_catalog(T, E)
EVENTS = events_between(E, EVAL_START, EVAL_END)
assert len(EVENTS) == REF['n_primary_events'], 'event catalog differs from notebook 5'
cal_rng = in_period(T, '2022-01-01', '2023-01-01'); ev_rng = in_period(T, EVAL_START, EVAL_END)
for name, rng in [('2021', in_period(T, '2021-01-01', '2022-01-01')), ('2022', cal_rng), ('2023-2026', ev_rng)]:
    print(f'{name}: spot-valid minutes {int((base_valid & rng).sum()):,} | common with futures {int((common & rng).sum()):,}')
display(xv.loc['2022'].describe().T.round(3))

spot  : 2020-01-01 00:00:00+00:00 -> 2026-04-30 23:59:00+00:00 3329280
futures: 2020-01-01 00:00:00+00:00 -> 2026-04-30 23:59:00+00:00 3329280
futures minutes available on the spot grid: 100.0000%
2021: spot-valid minutes 524,332 | common with futures 524,245
2022: spot-valid minutes 525,600 | common with futures 525,508
2023-2026: spot-valid minutes 1,750,857 | common with futures 1,750,500


,count,mean,std,min,25%,50%,75%,max
z_basis,525508.0,-0.023,1.088,-50.391,-0.648,-0.051,0.536,7.002
d_flow,525564.0,-0.005,0.062,-0.376,-0.043,-0.005,0.034,0.447
z_vr,525564.0,-0.174,1.078,-12.092,-0.858,-0.134,0.559,3.933


## 2. Reproduce the notebook 05 baseline (V+F) exactly

In [3]:
FEAT = {'z_vol': v, 'tsr': f, **{c: xv[c].to_numpy(float) for c in XV_FEATURES}}


def design(cols):
    return np.column_stack([FEAT[c] for c in cols])


tr0 = train_mask(T, base_valid)
m0 = new_logit().fit(design(['z_vol', 'tsr'])[tr0], y[tr0])
S0 = np.full(len(T), np.nan)
S0[base_valid] = m0.predict_proba(design(['z_vol', 'tsr'])[base_valid])[:, 1]
calm = base_valid & cal_rng; evm = base_valid & ev_rng
print('fast calendar identical to the reference on', check_fast_calendar(T[calm], S0[calm]), 'checks')
th0, n0, _ = calibrate_calendar(T[calm], S0[calm])
if abs(th0 - TH_CAL) > 1e-12:
    raise RuntimeError(f'baseline threshold {th0} != notebook-5 {TH_CAL}')
st0 = statistics(EVENTS, emit_calendar_fast(T[evm], S0[evm] >= th0))
if [st0['alarms'], st0['caught'], st0['before_1pct']] != REF['first_eligible']:
    raise RuntimeError(f'baseline {st0} != notebook-5 {REF["first_eligible"]}')
print('PASS: notebook-5 spot baseline reproduced', REF['first_eligible'], 'threshold', th0)

fast calendar identical to the reference on 14 checks
PASS: notebook-5 spot baseline reproduced [800, 39, 22] threshold 0.6698990591069133


## 3. Comparison of the five models (same minute set, same budget)

In [4]:
trc = train_mask(T, common); calc = common & cal_rng; evc = common & ev_rng
rows = []; alerts = {}; coef_rows = []; thresholds = {}
for name, cols in FEATURE_SETS.items():
    X = design(cols)
    m = new_logit().fit(X[trc], y[trc])
    th, ncal, _ = calibrate_calendar(T[calc], m.predict_proba(X[calc])[:, 1])
    s_ev = m.predict_proba(X[evc])[:, 1]
    a = emit_calendar_fast(T[evc], s_ev >= th)
    alerts[name] = a; thresholds[name] = th
    st = statistics(EVENTS, a)
    rows.append(dict(model=name, features='+'.join(cols), n_features=len(cols), threshold_2022=th, cal_alerts_2022=ncal, **st))
    coef_rows.append(dict(model=name, **{c: float(w) for c, w in zip(cols, m[-1].coef_[0])}))
    pd.DataFrame({'alarm_utc': pd.to_datetime(a, utc=True)}).to_csv(OUT / f'alarms_{name}.csv', index=False)
    match_events(EVENTS, a).to_csv(OUT / f'matched_{name}.csv', index=False)
res = pd.DataFrame(rows); res.to_csv(OUT / 'crossvenue_model_comparison.csv', index=False)
coefs = pd.DataFrame(coef_rows); coefs.to_csv(OUT / 'standardized_coefficients.csv', index=False)
display(res[['model', 'threshold_2022', 'cal_alerts_2022', 'alarms', 'caught', 'before_1pct', 'false_alarms', 'recall', 'pre_onset_recall']].round(4))
print('Standardized logistic coefficients (2021 fit):'); display(coefs.round(3))

pair_rows = [dict(challenger=k, **paired(EVENTS, alerts['M0_VF'], alerts[k])) for k in FEATURE_SETS if k != 'M0_VF']
pairs = pd.DataFrame(pair_rows); pairs.to_csv(OUT / 'paired_vs_M0.csv', index=False)
print('Paired event-level comparison against M0_VF on the same minute set (two-sided exact McNemar; exploratory):')
display(pairs.round(4))

yr = []
for k, a in alerts.items():
    for year in range(2023, 2027):
        ev_y = EVENTS[EVENTS.breach_utc.dt.year == year]
        a_y = a[pd.to_datetime(a, utc=True).year == year]
        s_ = statistics(ev_y, a_y) if len(ev_y) else dict(alarms=len(a_y), caught=0, before_1pct=0)
        yr.append(dict(model=k, year=year, alarms=len(a_y), events=len(ev_y), caught=s_['caught'], before_1pct=s_['before_1pct']))
yr = pd.DataFrame(yr); yr.to_csv(OUT / 'crossvenue_year_by_year.csv', index=False)
display(yr.pivot(index='year', columns='model', values='before_1pct'))

,model,threshold_2022,cal_alerts_2022,alarms,caught,before_1pct,false_alarms,recall,pre_onset_recall
0,M0_VF,0.6698,240,800,39,22,761,0.4699,0.2651
1,M1_VF_basis,0.6701,240,800,34,18,766,0.4096,0.2169
2,M2_VF_flowgap,0.6711,240,800,30,18,770,0.3614,0.2169
3,M3_VF_volratio,0.6765,240,800,31,17,769,0.3735,0.2048
4,M4_VF_all3,0.7191,240,798,33,13,765,0.3976,0.1566


Standardized logistic coefficients (2021 fit):


,model,z_vol,tsr,z_basis,d_flow,z_vr
0,M0_VF,0.511,0.022,NaN,NaN,NaN
1,M1_VF_basis,0.518,0.026,0.094,NaN,NaN
2,M2_VF_flowgap,0.513,0.032,NaN,0.066,NaN
3,M3_VF_volratio,0.473,0.022,NaN,NaN,0.241
4,M4_VF_all3,0.481,0.036,0.132,0.053,0.267


Paired event-level comparison against M0_VF on the same minute set (two-sided exact McNemar; exploratory):


,challenger,base_caught,new_caught,base_only_caught,new_only_caught,p_caught_two_sided,base_pre1pct,new_pre1pct,base_only_pre1pct,new_only_pre1pct,p_pre1pct_two_sided,p_pre1pct_one_sided_new_better
0,M1_VF_basis,39,34,6,1,0.1250,22,18,5,1,0.2188,0.9844
1,M2_VF_flowgap,39,30,9,0,0.0039,22,18,5,1,0.2188,0.9844
2,M3_VF_volratio,39,31,10,2,0.0386,22,17,7,2,0.1797,0.9805
3,M4_VF_all3,39,33,10,4,0.1796,22,13,10,1,0.0117,0.9995


model,M0_VF,M1_VF_basis,M2_VF_flowgap,M3_VF_volratio,M4_VF_all3
year,,,,,
2023,3,3,3,3,2
2024,7,4,5,5,4
2025,9,7,7,6,6
2026,3,4,3,3,1


## 4. Development-period (2021–2022) diagnostic: do feature values differ inside and outside the warning window?

In [5]:
dev = common & in_period(T, '2021-01-01', '2023-01-01')
diag = []
for c in ['z_vol', 'tsr'] + XV_FEATURES:
    x = FEAT[c][dev]; lab = y[dev] == 1
    sd = np.nanstd(x)
    diag.append(dict(feature=c, mean_in_warning_window=float(np.nanmean(x[lab])), mean_elsewhere=float(np.nanmean(x[~lab])),
                     standardized_difference=float((np.nanmean(x[lab]) - np.nanmean(x[~lab])) / sd) if sd > 0 else np.nan,
                     warning_minutes=int(lab.sum())))
diag = pd.DataFrame(diag); diag.to_csv(OUT / 'dev_feature_separation_2021_2022.csv', index=False)
display(diag.round(4))

,feature,mean_in_warning_window,mean_elsewhere,standardized_difference,warning_minutes
0,z_vol,0.5836,-0.0874,0.6270,30370
1,tsr,0.5099,0.5062,0.0282,30370
2,z_basis,0.0728,-0.0463,0.1041,30370
3,d_flow,-0.0035,-0.0045,0.0167,30370
4,z_vr,0.2960,-0.0514,0.3190,30370


## 5. Candidate selection by the pre-specified rule (passed to notebook 11)

In [6]:
base_row = res[res.model == 'M0_VF'].iloc[0]
ch = res[res.model != 'M0_VF'].sort_values(['before_1pct', 'n_features', 'model'], ascending=[False, True, True])
best = ch.iloc[0]
selected = str(best.model) if int(best.before_1pct) > int(base_row.before_1pct) else None
cand = {'selected': selected, 'rule': SPEC['candidate_rule'],
        'features': FEATURE_SETS[selected] if selected else None,
        'threshold_2022': float(thresholds[selected]) if selected else None,
        'baseline_common_mask': {'threshold_2022': float(thresholds['M0_VF']), 'alarms': int(base_row.alarms),
                                 'caught': int(base_row.caught), 'before_1pct': int(base_row.before_1pct)},
        'best_challenger': {'model': str(best.model), 'alarms': int(best.alarms), 'caught': int(best.caught),
                            'before_1pct': int(best.before_1pct)},
        'spec_sha256': hashlib.sha256(_raw).hexdigest()}
(OUT / 'crossvenue_candidate.json').write_text(json.dumps(cand, indent=2))
print(json.dumps(cand, indent=2))
print('\nExploratory only. If a candidate is selected, notebook 11 tests it once on 2019-11 → 2020-12 (futures data start 2019-09).')

{
  "selected": null,
  "rule": "challenger (M1-M4) with the most pre-onset hits; ties -> fewer features; passed to notebook 11 only if strictly more pre-onset hits than M0_VF on the same comparison mask, otherwise no candidate",
  "features": null,
  "threshold_2022": null,
  "baseline_common_mask": {
    "threshold_2022": 0.6697523129776579,
    "alarms": 800,
    "caught": 39,
    "before_1pct": 22
  },
  "best_challenger": {
    "model": "M1_VF_basis",
    "alarms": 800,
    "caught": 34,
    "before_1pct": 18
  },
  "spec_sha256": "a3561a711d906ef2b09d5a86cea218b5f9fb17e7709e96f911b613f2c45b4b0c"
}

Exploratory only. If a candidate is selected, notebook 11 tests it once on 2019-11 → 2020-12 (futures data start 2019-09).
